# SEO Reporting by Landing Page Type

**Approach:** Group GSC visibility metrics (impressions, clicks, CTR, weighted avg rank) by the `landing_page_type` already defined in the session table, then drill into which keywords drive each type.

**Why this works:**
- `landing_page_type` is deterministic from the URL -- no query classification needed
- GSC metrics aggregate cleanly at the page-type level (no grain mismatch)
- Session metrics (VC, funnel) are already segmentable by `landing_page_type`
- "Which keywords drive this page type?" is a simple drill-down

**Data sources:**
- `energy_prod.data_science.mp_session_level_query` -- session fact table with `landing_page_type`
- `lakehouse_production.common.gsc_search_analytics_d_1` -- Google Search Console data

In [ ]:
%sql
-- ====================================================================
-- STEP 1: URL-to-page-type lookup
-- ====================================================================
-- Builds a deduplicated mapping from normalized URL → landing_page_type.
-- Some URLs appear with multiple types across sessions; we pick the
-- most common type (by session count) for each URL.

CREATE OR REPLACE TEMP VIEW url_to_page_type AS

WITH ranked AS (
  SELECT
    RTRIM('/', LOWER(first_page_url)) AS landing_page,
    landing_page_type,
    SUM(sessions) AS total_sessions,
    ROW_NUMBER() OVER (
      PARTITION BY RTRIM('/', LOWER(first_page_url))
      ORDER BY SUM(sessions) DESC
    ) AS rn
  FROM energy_prod.data_science.mp_session_level_query
  WHERE _date >= '2025-01-01'
    AND landing_page_type IS NOT NULL
  GROUP BY RTRIM('/', LOWER(first_page_url)), landing_page_type
)

SELECT landing_page, landing_page_type
FROM ranked
WHERE rn = 1;

In [ ]:
%sql
-- Quick check: how many URLs per type?
SELECT landing_page_type, COUNT(*) AS url_count
FROM url_to_page_type
GROUP BY landing_page_type
ORDER BY url_count DESC;

In [ ]:
%sql
-- ====================================================================
-- STEP 2: GSC monthly metrics by site + landing_page_type
-- ====================================================================
-- Joins GSC rows to url_to_page_type, aggregates monthly.
-- Unmatched GSC pages (no session URL match) go to 'Unmatched'.

CREATE OR REPLACE TEMP VIEW gsc_monthly_by_page_type AS

WITH gsc_norm AS (
  SELECT
    DATE_TRUNC('month', g.date)  AS month,
    g.date,
    CASE g.domain
      WHEN 'choosetexaspower.org'  THEN 'CTXP'
      WHEN 'saveonenergy.com'      THEN 'SOE'
      WHEN 'chooseenergy.com'      THEN 'Choose TX'
      WHEN 'texaselectricrates.com' THEN 'TXER'
    END AS site,
    RTRIM('/',
      LOWER(
        CASE WHEN POSITION('#' IN g.page) > 0
          THEN LEFT(g.page, POSITION('#' IN g.page) - 1)
          ELSE g.page
        END
      )
    ) AS landing_page,
    g.clicks,
    g.impressions,
    g.position
  FROM lakehouse_production.common.gsc_search_analytics_d_1 g
  WHERE g.date >= '2024-01-01'
    AND g.domain IN (
      'choosetexaspower.org','saveonenergy.com',
      'chooseenergy.com','texaselectricrates.com'
    )
)

SELECT
  gn.month,
  gn.site,
  COUNT(DISTINCT gn.date)                                               AS days_with_data,
  DAY(LAST_DAY(gn.month))                                              AS days_in_month,
  COALESCE(u.landing_page_type, 'Unmatched')                           AS landing_page_type,
  SUM(gn.clicks)                                                        AS clicks,
  SUM(gn.impressions)                                                   AS impressions,
  ROUND(SUM(gn.clicks) * 100.0 / NULLIF(SUM(gn.impressions), 0), 3)   AS ctr_pct,
  ROUND(
    SUM(gn.position * gn.impressions) / NULLIF(SUM(gn.impressions), 0), 1
  )                                                                     AS weighted_avg_rank
FROM gsc_norm gn
LEFT JOIN url_to_page_type u ON gn.landing_page = u.landing_page
GROUP BY gn.month, gn.site, COALESCE(u.landing_page_type, 'Unmatched'),
         DAY(LAST_DAY(gn.month))
ORDER BY gn.month, site, clicks DESC;

In [ ]:
%sql
-- Sanity check: total clicks by site + page type (all time)
SELECT
  site,
  landing_page_type,
  SUM(clicks) AS total_clicks,
  SUM(impressions) AS total_impressions,
  ROUND(SUM(clicks) * 100.0 / NULLIF(SUM(impressions), 0), 3) AS ctr_pct,
  ROUND(
    SUM(CASE WHEN clicks > 0 THEN weighted_avg_rank * clicks ELSE 0 END)
    / NULLIF(SUM(CASE WHEN clicks > 0 THEN clicks ELSE 0 END), 0), 1
  ) AS click_weighted_rank
FROM gsc_monthly_by_page_type
GROUP BY site, landing_page_type
ORDER BY site, total_clicks DESC;

---
## Monthly Trend Charts by Site + Landing Page Type

4-panel chart: Impressions, Clicks, CTR (%), and Weighted Avg Rank.

One line per major page type. Partial months (April 2026) are paced to full-month equivalents.

**Site filter:** Set `SITE_FILTER` to `'All'` for combined view, or `'CTXP'`, `'SOE'`, `'Choose TX'`, `'TXER'` for a single site.

In [ ]:
%python
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# ---- SITE FILTER: 'All', 'CTXP', 'SOE', 'Choose TX', or 'TXER' ----
SITE_FILTER = 'All'

df = spark.sql("SELECT * FROM gsc_monthly_by_page_type").toPandas()
df['month'] = pd.to_datetime(df['month']).dt.tz_localize(None)
df['days_with_data'] = df['days_with_data'].astype(int)
df['days_in_month'] = df['days_in_month'].astype(int)
df['is_partial'] = df['days_with_data'] < df['days_in_month']

if SITE_FILTER != 'All':
    df = df[df['site'] == SITE_FILTER]

# Re-aggregate across sites when showing All (sum clicks/impressions, recalc CTR/rank)
agg = df.groupby(['month', 'landing_page_type', 'days_in_month']).agg(
    days_with_data=('days_with_data', 'max'),
    clicks=('clicks', 'sum'),
    impressions=('impressions', 'sum'),
    weighted_pos_x_impr=('impressions', lambda x: (df.loc[x.index, 'weighted_avg_rank'] * df.loc[x.index, 'impressions']).sum())
).reset_index()
agg['ctr_pct'] = (agg['clicks'] * 100.0 / agg['impressions'].replace(0, pd.NA)).round(3)
agg['weighted_avg_rank'] = (agg['weighted_pos_x_impr'] / agg['impressions'].replace(0, pd.NA)).round(1)
agg['is_partial'] = agg['days_with_data'] < agg['days_in_month']
agg['clicks_paced'] = agg.apply(
    lambda r: r['clicks'] * r['days_in_month'] / r['days_with_data'] if r['is_partial'] else r['clicks'], axis=1
)
agg['impressions_paced'] = agg.apply(
    lambda r: r['impressions'] * r['days_in_month'] / r['days_with_data'] if r['is_partial'] else r['impressions'], axis=1
)

top_types = (
    agg[agg['landing_page_type'] != 'Unmatched']
    .groupby('landing_page_type')['clicks']
    .sum()
    .nlargest(10)
    .index.tolist()
)

colors = [
    '#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
    '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52'
]

site_label = SITE_FILTER if SITE_FILTER != 'All' else 'All Sites'
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Monthly Impressions (paced)', 'Monthly Clicks (paced)',
                    'CTR (%)', 'Weighted Avg Rank (lower = better)'),
    vertical_spacing=0.12, horizontal_spacing=0.08
)

for i, lpt in enumerate(top_types):
    sub = agg[agg['landing_page_type'] == lpt].sort_values('month')
    color = colors[i % len(colors)]

    fig.add_trace(go.Scatter(
        x=sub['month'], y=sub['impressions_paced'], name=lpt,
        mode='lines+markers', line=dict(color=color, width=2), marker=dict(size=4),
        legendgroup=lpt, showlegend=True
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=sub['month'], y=sub['clicks_paced'], name=lpt,
        mode='lines+markers', line=dict(color=color, width=2), marker=dict(size=4),
        legendgroup=lpt, showlegend=False
    ), row=1, col=2)

    fig.add_trace(go.Scatter(
        x=sub['month'], y=sub['ctr_pct'], name=lpt,
        mode='lines+markers', line=dict(color=color, width=2), marker=dict(size=4),
        legendgroup=lpt, showlegend=False
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=sub['month'], y=sub['weighted_avg_rank'], name=lpt,
        mode='lines+markers', line=dict(color=color, width=2), marker=dict(size=4),
        legendgroup=lpt, showlegend=False
    ), row=2, col=2)

fig.update_yaxes(autorange='reversed', row=2, col=2)
fig.update_layout(
    height=700, width=1100,
    title_text=f'SEO Visibility by Landing Page Type — {site_label} (Jan 2024 – Apr 2026)',
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5,
                font=dict(size=10)),
    margin=dict(b=120)
)
for r in range(1, 3):
    for c in range(1, 3):
        fig.update_xaxes(dtick='M3', tickformat='%b %Y', row=r, col=c)

fig.show()

---
## April vs March Diagnostic by Page Type

Compares April MTD (paced) vs March for each page type: which types lost the most impressions, clicks, or rank?

In [ ]:
%sql
-- ====================================================================
-- April vs March comparison by site + page type (same day count: 1st-10th)
-- ====================================================================

CREATE OR REPLACE TEMP VIEW page_type_apr_vs_mar AS

WITH gsc_norm AS (
  SELECT
    g.date,
    CASE g.domain
      WHEN 'choosetexaspower.org'  THEN 'CTXP'
      WHEN 'saveonenergy.com'      THEN 'SOE'
      WHEN 'chooseenergy.com'      THEN 'Choose TX'
      WHEN 'texaselectricrates.com' THEN 'TXER'
    END AS site,
    RTRIM('/',
      LOWER(
        CASE WHEN POSITION('#' IN g.page) > 0
          THEN LEFT(g.page, POSITION('#' IN g.page) - 1)
          ELSE g.page
        END
      )
    ) AS landing_page,
    g.clicks, g.impressions, g.position
  FROM lakehouse_production.common.gsc_search_analytics_d_1 g
  WHERE g.date >= '2026-03-01' AND g.date <= '2026-04-10'
    AND g.domain IN (
      'choosetexaspower.org','saveonenergy.com',
      'chooseenergy.com','texaselectricrates.com'
    )
),

tagged AS (
  SELECT
    CASE
      WHEN gn.date BETWEEN '2026-04-01' AND '2026-04-10' THEN 'April'
      WHEN gn.date BETWEEN '2026-03-01' AND '2026-03-10' THEN 'March'
    END AS period,
    gn.site,
    COALESCE(u.landing_page_type, 'Unmatched') AS landing_page_type,
    gn.clicks, gn.impressions, gn.position
  FROM gsc_norm gn
  LEFT JOIN url_to_page_type u ON gn.landing_page = u.landing_page
  WHERE gn.date BETWEEN '2026-04-01' AND '2026-04-10'
     OR gn.date BETWEEN '2026-03-01' AND '2026-03-10'
),

mar AS (
  SELECT site, landing_page_type,
    SUM(clicks) AS clicks, SUM(impressions) AS impressions,
    ROUND(SUM(clicks)*100.0/NULLIF(SUM(impressions),0),3) AS ctr_pct,
    ROUND(SUM(position*impressions)/NULLIF(SUM(impressions),0),1) AS avg_rank
  FROM tagged WHERE period = 'March' GROUP BY site, landing_page_type
),
apr AS (
  SELECT site, landing_page_type,
    SUM(clicks) AS clicks, SUM(impressions) AS impressions,
    ROUND(SUM(clicks)*100.0/NULLIF(SUM(impressions),0),3) AS ctr_pct,
    ROUND(SUM(position*impressions)/NULLIF(SUM(impressions),0),1) AS avg_rank
  FROM tagged WHERE period = 'April' GROUP BY site, landing_page_type
)

SELECT
  COALESCE(m.site, a.site)                             AS site,
  COALESCE(m.landing_page_type, a.landing_page_type)   AS landing_page_type,
  COALESCE(m.clicks, 0)       AS mar_clicks,
  COALESCE(a.clicks, 0)       AS apr_clicks,
  COALESCE(a.clicks, 0) - COALESCE(m.clicks, 0) AS click_delta,
  ROUND((COALESCE(a.clicks,0) - COALESCE(m.clicks,0)) * 100.0
    / NULLIF(COALESCE(m.clicks,0), 0), 1) AS click_delta_pct,
  COALESCE(m.impressions, 0)  AS mar_impressions,
  COALESCE(a.impressions, 0)  AS apr_impressions,
  ROUND((COALESCE(a.impressions,0) - COALESCE(m.impressions,0)) * 100.0
    / NULLIF(COALESCE(m.impressions,0), 0), 1) AS impr_delta_pct,
  m.ctr_pct AS mar_ctr, a.ctr_pct AS apr_ctr,
  m.avg_rank AS mar_rank, a.avg_rank AS apr_rank
FROM mar m
FULL OUTER JOIN apr a ON m.site = a.site AND m.landing_page_type = a.landing_page_type
ORDER BY site, click_delta ASC;

SELECT * FROM page_type_apr_vs_mar;

In [ ]:
%python
# ---- SITE FILTER: 'All', 'CTXP', 'SOE', 'Choose TX', or 'TXER' ----
SITE_FILTER = 'All'

comp = spark.sql("SELECT * FROM page_type_apr_vs_mar").toPandas()

if SITE_FILTER != 'All':
    comp = comp[comp['site'] == SITE_FILTER]
    # Already at site-level, no need to re-aggregate
else:
    # Aggregate across all sites per landing_page_type
    comp = comp.groupby('landing_page_type').agg(
        mar_clicks=('mar_clicks', 'sum'),
        apr_clicks=('apr_clicks', 'sum'),
        click_delta=('click_delta', 'sum'),
        mar_impressions=('mar_impressions', 'sum'),
        apr_impressions=('apr_impressions', 'sum'),
    ).reset_index()
    comp['click_delta_pct'] = (comp['click_delta'] * 100.0 / comp['mar_clicks'].replace(0, pd.NA)).round(1)
    comp['impr_delta_pct'] = ((comp['apr_impressions'] - comp['mar_impressions']) * 100.0
                               / comp['mar_impressions'].replace(0, pd.NA)).round(1)

comp = comp[(comp['mar_clicks'] >= 20) | (comp['apr_clicks'] >= 20)].copy()
comp = comp.sort_values('click_delta')

site_label = SITE_FILTER if SITE_FILTER != 'All' else 'All Sites'

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    y=comp['landing_page_type'],
    x=comp['click_delta'],
    orientation='h',
    marker_color=['#D62728' if v < 0 else '#2CA02C' for v in comp['click_delta']],
    text=[f"{v:+,.0f} ({p:+.1f}%)" if pd.notna(p) else f"{v:+,.0f}"
          for v, p in zip(comp['click_delta'], comp['click_delta_pct'])],
    textposition='outside'
))
fig2.update_layout(
    title=f'GSC Click Change by Page Type — {site_label} (Apr 1-10 vs Mar 1-10)',
    xaxis_title='Click Change',
    height=550, margin=dict(l=180)
)
fig2.show()

# Impression change
comp_i = comp.sort_values('impr_delta_pct')
fig2b = go.Figure()
fig2b.add_trace(go.Bar(
    y=comp_i['landing_page_type'],
    x=comp_i['impr_delta_pct'],
    orientation='h',
    marker_color=['#D62728' if v < 0 else '#2CA02C' for v in comp_i['impr_delta_pct']],
    text=[f"{v:+.1f}%" if pd.notna(v) else '' for v in comp_i['impr_delta_pct']],
    textposition='outside'
))
fig2b.update_layout(
    title=f'GSC Impression Change % by Page Type — {site_label} (Apr 1-10 vs Mar 1-10)',
    xaxis_title='Impression Change %',
    height=550, margin=dict(l=180)
)
fig2b.show()

In [ ]:
%python
# Rank change bubble chart
# This chart works best at the site level; at All Sites we use weighted rank from the full data
comp_full = spark.sql("SELECT * FROM page_type_apr_vs_mar").toPandas()

if SITE_FILTER != 'All':
    comp_bub = comp_full[comp_full['site'] == SITE_FILTER].copy()
else:
    comp_bub = comp_full.groupby('landing_page_type').agg(
        mar_clicks=('mar_clicks', 'sum'),
        apr_clicks=('apr_clicks', 'sum'),
        click_delta=('click_delta', 'sum'),
        mar_impressions=('mar_impressions', 'sum'),
        apr_impressions=('apr_impressions', 'sum'),
        mar_rank=('mar_rank', 'mean'),
        apr_rank=('apr_rank', 'mean'),
    ).reset_index()
    comp_bub['click_delta_pct'] = (comp_bub['click_delta'] * 100.0 / comp_bub['mar_clicks'].replace(0, pd.NA)).round(1)
    comp_bub['impr_delta_pct'] = ((comp_bub['apr_impressions'] - comp_bub['mar_impressions']) * 100.0
                                   / comp_bub['mar_impressions'].replace(0, pd.NA)).round(1)

comp_bub = comp_bub[(comp_bub['mar_clicks'] >= 20) | (comp_bub['apr_clicks'] >= 20)].copy()
comp_bub['rank_delta'] = comp_bub['apr_rank'].astype(float) - comp_bub['mar_rank'].astype(float)

fig2c = go.Figure()
fig2c.add_trace(go.Scatter(
    x=comp_bub['impr_delta_pct'].astype(float),
    y=comp_bub['rank_delta'],
    mode='markers+text',
    marker=dict(
        size=comp_bub['apr_clicks'].astype(float).clip(lower=20) / comp_bub['apr_clicks'].astype(float).max() * 50 + 8,
        color=comp_bub['click_delta_pct'].astype(float),
        colorscale='RdYlGn',
        colorbar=dict(title='Click delta %'),
        line=dict(width=1, color='DarkSlateGrey')
    ),
    text=comp_bub['landing_page_type'],
    textposition='top center',
    textfont=dict(size=9)
))
fig2c.update_layout(
    title=f'Page Type Health: Rank vs Impression Change — {site_label} (Apr vs Mar)',
    xaxis_title='Impression Change %',
    yaxis_title='Avg Rank Change (positive = worse)',
    height=500,
    yaxis=dict(autorange='reversed'),
    shapes=[
        dict(type='line', x0=0, x1=0, y0=-5, y1=5, line=dict(dash='dash', color='grey')),
        dict(type='line', x0=-80, x1=80, y0=0, y1=0, line=dict(dash='dash', color='grey'))
    ]
)
fig2c.show()

---
## Top Queries per Landing Page Type

For each major page type, which GSC queries drive the most clicks? This shows the keyword intent behind each page category.

In [ ]:
%sql
-- ====================================================================
-- Top 10 queries per site + landing_page_type (by clicks, April 2026)
-- ====================================================================

CREATE OR REPLACE TEMP VIEW top_queries_by_page_type AS

WITH gsc_norm AS (
  SELECT
    CASE g.domain
      WHEN 'choosetexaspower.org'  THEN 'CTXP'
      WHEN 'saveonenergy.com'      THEN 'SOE'
      WHEN 'chooseenergy.com'      THEN 'Choose TX'
      WHEN 'texaselectricrates.com' THEN 'TXER'
    END AS site,
    RTRIM('/',
      LOWER(
        CASE WHEN POSITION('#' IN g.page) > 0
          THEN LEFT(g.page, POSITION('#' IN g.page) - 1)
          ELSE g.page
        END
      )
    ) AS landing_page,
    LOWER(TRIM(g.query)) AS query,
    g.clicks, g.impressions, g.position
  FROM lakehouse_production.common.gsc_search_analytics_d_1 g
  WHERE g.date >= '2026-04-01' AND g.date <= '2026-04-10'
    AND g.domain IN (
      'choosetexaspower.org','saveonenergy.com',
      'chooseenergy.com','texaselectricrates.com'
    )
),

tagged AS (
  SELECT
    gn.site,
    COALESCE(u.landing_page_type, 'Unmatched') AS landing_page_type,
    gn.query,
    SUM(gn.clicks) AS clicks,
    SUM(gn.impressions) AS impressions,
    ROUND(SUM(gn.position * gn.impressions) / NULLIF(SUM(gn.impressions), 0), 1) AS avg_rank
  FROM gsc_norm gn
  LEFT JOIN url_to_page_type u ON gn.landing_page = u.landing_page
  GROUP BY gn.site, COALESCE(u.landing_page_type, 'Unmatched'), gn.query
),

ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY site, landing_page_type ORDER BY clicks DESC) AS rn
  FROM tagged
)

SELECT site, landing_page_type, query, clicks, impressions, avg_rank
FROM ranked
WHERE rn <= 10
ORDER BY site, landing_page_type, clicks DESC;

In [ ]:
%python
# ---- SITE FILTER: 'All', 'CTXP', 'SOE', 'Choose TX', or 'TXER' ----
SITE_FILTER = 'All'

tq = spark.sql("SELECT * FROM top_queries_by_page_type").toPandas()

if SITE_FILTER != 'All':
    tq = tq[tq['site'] == SITE_FILTER]

key_types = ['Homepage', 'StateGEO', 'Tier1CityGEO', 'Tier2CityGEO',
             'Provider', 'No_Deposit_Plans', 'PowerToChoose',
             'Solar_Buyback_Plans', 'Free_Nights_Weekends_Plans']

site_label = SITE_FILTER if SITE_FILTER != 'All' else 'All Sites'
for lpt in key_types:
    sub = tq[tq['landing_page_type'] == lpt].sort_values('clicks', ascending=False).head(10)
    if sub.empty:
        continue
    print(f"\n{'='*70}")
    print(f"  [{site_label}] {lpt}  (top 10 queries by clicks, Apr 1-10)")
    print(f"{'='*70}")
    for _, r in sub.iterrows():
        site_col = f" [{r['site']}]" if SITE_FILTER == 'All' else ''
        print(f"  {int(r['clicks']):>4} clicks | {int(r['impressions']):>6} impr | rank {float(r['avg_rank']):>5.1f} |{site_col} {r['query']}")

---
## Session Metrics by Landing Page Type

Funnel performance by page type -- entirely from the session table (no GSC join needed).

In [ ]:
%sql
-- ====================================================================
-- Session funnel by site + landing_page_type (Organic, April 2026)
-- ====================================================================

CREATE OR REPLACE TEMP VIEW session_funnel_by_page_type AS

SELECT
  website AS site,
  landing_page_type,
  SUM(sessions)                                AS sessions,
  SUM(zip_entry)                               AS zip_entries,
  SUM(is_product_viewed)                       AS product_views,
  SUM(has_cart)                                 AS carts,
  SUM(cart_orders) + SUM(phone_orders)         AS orders,
  ROUND(SUM(zip_entry) * 100.0 / NULLIF(SUM(sessions), 0), 1)          AS zlur_pct,
  ROUND(SUM(has_cart) * 100.0 / NULLIF(SUM(sessions), 0), 1)           AS cart_rate_pct,
  ROUND((SUM(cart_orders) + SUM(phone_orders)) * 100.0
    / NULLIF(SUM(sessions), 0), 2)             AS vc_pct
FROM energy_prod.data_science.mp_session_level_query
WHERE marketing_channel = 'Organic'
  AND _date >= '2026-04-01'
GROUP BY
  website,
  landing_page_type
HAVING SUM(sessions) >= 10
ORDER BY site, sessions DESC;

SELECT * FROM session_funnel_by_page_type;

In [ ]:
%sql
-- ====================================================================
-- Monthly session trend by site + landing_page_type (Organic)
-- ====================================================================

CREATE OR REPLACE TEMP VIEW session_monthly_by_page_type AS

SELECT
  DATE_TRUNC('month', _date)    AS month,
  COUNT(DISTINCT _date)          AS days_with_data,
  website                        AS site,
  landing_page_type,
  SUM(sessions)                  AS sessions,
  SUM(cart_orders) + SUM(phone_orders) AS orders,
  ROUND((SUM(cart_orders) + SUM(phone_orders)) * 100.0
    / NULLIF(SUM(sessions), 0), 2) AS vc_pct
FROM energy_prod.data_science.mp_session_level_query
WHERE marketing_channel = 'Organic'
  AND _date >= '2025-01-01'
GROUP BY
  DATE_TRUNC('month', _date),
  website,
  landing_page_type
ORDER BY month, site, sessions DESC;

In [ ]:
%python
# ---- SITE FILTER: 'All', 'CTXP', 'SOE', 'Choose TX', or 'TXER' ----
SITE_FILTER = 'All'

sess = spark.sql("SELECT * FROM session_monthly_by_page_type").toPandas()
sess['month'] = pd.to_datetime(sess['month']).dt.tz_localize(None)
sess['days_with_data'] = sess['days_with_data'].astype(int)

if SITE_FILTER != 'All':
    sess = sess[sess['site'] == SITE_FILTER]

# Re-aggregate across sites when showing All
sess_agg = sess.groupby(['month', 'landing_page_type']).agg(
    days_with_data=('days_with_data', 'max'),
    sessions=('sessions', 'sum'),
    orders=('orders', 'sum'),
).reset_index()
sess_agg['vc_pct'] = (sess_agg['orders'] * 100.0 / sess_agg['sessions'].replace(0, pd.NA)).round(2)

import calendar
sess_agg['days_in_month'] = sess_agg['month'].apply(lambda m: calendar.monthrange(m.year, m.month)[1])
sess_agg['sessions_paced'] = sess_agg.apply(
    lambda r: r['sessions'] * r['days_in_month'] / r['days_with_data']
      if r['days_with_data'] < r['days_in_month'] else r['sessions'], axis=1
)

top_sess_types = (
    sess_agg.groupby('landing_page_type')['sessions'].sum()
    .nlargest(10).index.tolist()
)

site_label = SITE_FILTER if SITE_FILTER != 'All' else 'All Sites'

fig3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Organic Sessions by Page Type (paced)', 'VC % by Page Type'),
    horizontal_spacing=0.08
)

for i, lpt in enumerate(top_sess_types):
    sub = sess_agg[sess_agg['landing_page_type'] == lpt].sort_values('month')
    color = colors[i % len(colors)]

    fig3.add_trace(go.Scatter(
        x=sub['month'], y=sub['sessions_paced'], name=lpt,
        mode='lines+markers', line=dict(color=color, width=2), marker=dict(size=4),
        legendgroup=lpt, showlegend=True
    ), row=1, col=1)

    fig3.add_trace(go.Scatter(
        x=sub['month'], y=sub['vc_pct'], name=lpt,
        mode='lines+markers', line=dict(color=color, width=2), marker=dict(size=4),
        legendgroup=lpt, showlegend=False
    ), row=1, col=2)

fig3.update_layout(
    height=500, width=1100,
    title_text=f'Organic Sessions & VC by Page Type — {site_label} Monthly Trend',
    legend=dict(orientation='h', yanchor='bottom', y=-0.2, xanchor='center', x=0.5,
                font=dict(size=10)),
    margin=dict(b=120)
)
for c in range(1, 3):
    fig3.update_xaxes(dtick='M3', tickformat='%b %Y', row=1, col=c)
fig3.update_yaxes(title_text='Sessions', row=1, col=1)
fig3.update_yaxes(title_text='VC %', row=1, col=2)
fig3.show()

---
## Combined View: GSC Visibility + Session Performance

Side-by-side: GSC metrics (aggregated from GSC directly) alongside session metrics (from session table) for April 2026.

In [ ]:
%sql
-- ====================================================================
-- Combined: GSC visibility + session funnel by site + page type (April)
-- ====================================================================

SELECT
  s.site,
  s.landing_page_type,
  s.sessions,
  s.orders,
  s.vc_pct,
  g.apr_clicks    AS gsc_clicks,
  g.apr_impressions AS gsc_impressions,
  g.apr_ctr       AS gsc_ctr_pct,
  g.apr_rank      AS gsc_avg_rank,
  g.click_delta,
  g.click_delta_pct,
  g.impr_delta_pct
FROM session_funnel_by_page_type s
LEFT JOIN page_type_apr_vs_mar g
  ON s.site = g.site
  AND s.landing_page_type = g.landing_page_type
ORDER BY s.site, s.sessions DESC;

---
## Finalized Query: Sessions + GSC in One View

This is the single self-contained query you can drop into a Databricks dashboard. It aggregates GSC metrics and session metrics **separately** by `landing_page_type` and `month`, then joins the two summaries. This guarantees no double counting.

**How it works:**
1. `url_to_page_type` -- maps normalized URLs to their page type (from session table, deduped)
2. `gsc_agg` -- aggregates impressions, clicks, CTR, rank from GSC by month + page type
3. `session_agg` -- aggregates sessions, orders, VC from session table by month + page type
4. Final SELECT joins the two on `month + landing_page_type`

**Columns produced:**
- `month`, `site`, `landing_page_type`
- `gsc_impressions`, `gsc_clicks`, `gsc_ctr_pct`, `gsc_weighted_avg_rank` -- from GSC table directly
- `sessions`, `orders`, `vc_pct` -- from session table directly
- `days_with_gsc_data`, `days_in_month` -- for pacing partial months

**Site mapping:** GSC `domain` is mapped to match session `website` values:
| site | GSC domain |
|------|-----------|
| CTXP | choosetexaspower.org |
| SOE | saveonenergy.com |
| Choose TX | chooseenergy.com |
| TXER | texaselectricrates.com |

In [ ]:
%sql
-- ====================================================================
-- FINALIZED QUERY: Monthly sessions + GSC by site + landing_page_type
-- ====================================================================
-- Drop this into a Databricks dashboard or save as a view.
-- GSC and sessions are aggregated SEPARATELY then joined,
-- so impressions/clicks are never double-counted.

WITH url_to_page_type AS (
  -- Deduplicated URL → page type mapping.
  -- Picks the most common type per URL (by session volume).
  SELECT landing_page, landing_page_type
  FROM (
    SELECT
      RTRIM('/', LOWER(first_page_url))  AS landing_page,
      landing_page_type,
      ROW_NUMBER() OVER (
        PARTITION BY RTRIM('/', LOWER(first_page_url))
        ORDER BY SUM(sessions) DESC
      ) AS rn
    FROM energy_prod.data_science.mp_session_level_query
    WHERE _date >= '2025-01-01'
      AND landing_page_type IS NOT NULL
    GROUP BY RTRIM('/', LOWER(first_page_url)), landing_page_type
  )
  WHERE rn = 1
),

-- GSC side: aggregate by month + site + page type
gsc_agg AS (
  SELECT
    DATE_TRUNC('month', g.date)                                           AS month,
    COUNT(DISTINCT g.date)                                                AS days_with_gsc_data,
    DAY(LAST_DAY(DATE_TRUNC('month', g.date)))                           AS days_in_month,
    -- Map GSC domain to match session website values
    CASE g.domain
      WHEN 'choosetexaspower.org'  THEN 'CTXP'
      WHEN 'saveonenergy.com'      THEN 'SOE'
      WHEN 'chooseenergy.com'      THEN 'Choose TX'
      WHEN 'texaselectricrates.com' THEN 'TXER'
    END                                                                   AS site,
    COALESCE(u.landing_page_type, 'Unmatched')                           AS landing_page_type,
    SUM(g.clicks)                                                         AS gsc_clicks,
    SUM(g.impressions)                                                    AS gsc_impressions,
    ROUND(SUM(g.clicks) * 100.0 / NULLIF(SUM(g.impressions), 0), 3)     AS gsc_ctr_pct,
    ROUND(
      SUM(g.position * g.impressions) / NULLIF(SUM(g.impressions), 0), 1
    )                                                                     AS gsc_weighted_avg_rank
  FROM lakehouse_production.common.gsc_search_analytics_d_1 g
  LEFT JOIN url_to_page_type u
    ON RTRIM('/',
         LOWER(
           CASE WHEN POSITION('#' IN g.page) > 0
             THEN LEFT(g.page, POSITION('#' IN g.page) - 1)
             ELSE g.page
           END
         )
       ) = u.landing_page
  WHERE g.date >= '2025-01-01'
    AND g.domain IN (
      'choosetexaspower.org', 'saveonenergy.com',
      'chooseenergy.com', 'texaselectricrates.com'
    )
  GROUP BY
    DATE_TRUNC('month', g.date),
    DAY(LAST_DAY(DATE_TRUNC('month', g.date))),
    CASE g.domain
      WHEN 'choosetexaspower.org'  THEN 'CTXP'
      WHEN 'saveonenergy.com'      THEN 'SOE'
      WHEN 'chooseenergy.com'      THEN 'Choose TX'
      WHEN 'texaselectricrates.com' THEN 'TXER'
    END,
    COALESCE(u.landing_page_type, 'Unmatched')
),

-- Session side: aggregate by month + site + page type
session_agg AS (
  SELECT
    DATE_TRUNC('month', _date)                AS month,
    website                                    AS site,
    landing_page_type,
    SUM(sessions)                              AS sessions,
    SUM(zip_entry)                             AS zip_entries,
    SUM(has_cart)                               AS carts,
    SUM(cart_orders) + SUM(phone_orders)       AS orders,
    ROUND(
      (SUM(cart_orders) + SUM(phone_orders)) * 100.0
      / NULLIF(SUM(sessions), 0), 2
    )                                          AS vc_pct
  FROM energy_prod.data_science.mp_session_level_query
  WHERE _date >= '2025-01-01'
    AND marketing_channel = 'Organic'
  GROUP BY
    DATE_TRUNC('month', _date),
    website,
    landing_page_type
)

-- Final join: one row per month + site + page type
SELECT
  COALESCE(g.month, s.month)                         AS month,
  COALESCE(g.site, s.site)                           AS site,
  COALESCE(g.landing_page_type, s.landing_page_type) AS landing_page_type,

  -- GSC visibility (from GSC table)
  g.gsc_impressions,
  g.gsc_clicks,
  g.gsc_ctr_pct,
  g.gsc_weighted_avg_rank,

  -- Session performance (from session table)
  s.sessions,
  s.zip_entries,
  s.carts,
  s.orders,
  s.vc_pct,

  -- Pacing helpers
  g.days_with_gsc_data,
  g.days_in_month

FROM gsc_agg g
FULL OUTER JOIN session_agg s
  ON g.month = s.month
  AND g.site = s.site
  AND g.landing_page_type = s.landing_page_type

ORDER BY month DESC, site, COALESCE(s.sessions, 0) DESC;

---
## Validation Guide

### Validation checks to run

**1. Total GSC clicks should match the raw table (overall and per site)**
```sql
-- Overall: should equal SUM(gsc_clicks) from the finalized query for the same month
SELECT SUM(clicks) FROM lakehouse_production.common.gsc_search_analytics_d_1
WHERE date BETWEEN '2026-04-01' AND '2026-04-10'
  AND domain IN ('choosetexaspower.org','saveonenergy.com','chooseenergy.com','texaselectricrates.com');

-- Per site: compare to finalized query grouped by site
SELECT domain, SUM(clicks) FROM lakehouse_production.common.gsc_search_analytics_d_1
WHERE date BETWEEN '2026-04-01' AND '2026-04-10'
  AND domain IN ('choosetexaspower.org','saveonenergy.com','chooseenergy.com','texaselectricrates.com')
GROUP BY domain;
```
If the finalized query total is higher, you have a URL mapping to multiple page types (the dedup should prevent this). If lower, some GSC pages aren't joining.

**2. Total sessions should match the raw table (overall and per site)**
```sql
-- Overall
SELECT SUM(sessions) FROM energy_prod.data_science.mp_session_level_query
WHERE marketing_channel = 'Organic' AND _date BETWEEN '2026-04-01' AND '2026-04-13';

-- Per site: compare to finalized query grouped by site
SELECT website, SUM(sessions) FROM energy_prod.data_science.mp_session_level_query
WHERE marketing_channel = 'Organic' AND _date BETWEEN '2026-04-01' AND '2026-04-13'
GROUP BY website;
```

**3. Spot-check a single site + page type**

Pick a site like `CTXP` and a page type like `Tier1CityGEO`, manually sum clicks for its known URLs in GSC, and compare to the query output.

### Pitfalls to avoid in Databricks dashboards

**Double counting impressions/clicks:**
- The finalized query is safe because GSC and sessions are aggregated **separately** then joined.
- If you ever join GSC rows directly to session rows (one-to-many), then SUM(impressions) will overcount. Never do this.
- Rule of thumb: **if your FROM clause has the session table, only SUM session columns.** If your FROM clause has the GSC table, only SUM GSC columns.

**Partial month pacing:**
- April 2026 will have fewer days of GSC data than sessions (GSC lags ~2 days).
- Use the `days_with_gsc_data` and `days_in_month` columns to pace: `gsc_clicks * days_in_month / days_with_gsc_data`.
- Do NOT pace sessions using GSC day counts -- use the session table's own day count.

**"Unmatched" page type:**
- GSC pages that don't appear in any organic session get `landing_page_type = 'Unmatched'`. This is expected -- GSC tracks pages that get impressions even if nobody clicks through.
- These will have GSC metrics but NULL sessions. Don't panic about NULL sessions on Unmatched rows.

**Session-only page types:**
- Some page types exist in sessions but not in GSC (e.g., `Cart` -- people don't Google their way to a cart URL).
- These will have sessions but NULL GSC metrics. This is correct.

**CTR looks very low (0.1-0.3%):**
- This is normal for GSC. GSC counts an "impression" every time your page appears in any search result, even page 5. Most impressions never get seen by the user.
- CTR is most useful as a **relative** metric (comparing page types or months), not as an absolute number.

**Weighted avg rank != your "real" position:**
- GSC `position` is the highest position your page achieved for that query in that session. If you appeared at #3 and #8, it records 3.
- The weighted average is dominated by high-impression queries, which tend to be broader and more competitive (higher position numbers).
- Use rank for **directional trends** (is it getting better or worse?), not as an exact position.

**Site mapping between GSC and sessions:**
- GSC uses `domain` (e.g. `choosetexaspower.org`) which is mapped to the session `website` values (CTXP, SOE, Choose TX, TXER).
- The session side uses `website` directly with no transformation.
- The `Unknown` website value in session data (rare, ~2 sessions/month) passes through as-is. These won't join to any GSC data, which is fine.
- When filtering by site in a dashboard, filter on the `site` column using the session website names: `CTXP`, `SOE`, `Choose TX`, `TXER`.